# List — brapi-py

Interactive notebook for exploring the BrAPI **List** endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).  
> If `brapi-py` is not installed, uncomment the `pip install` line in the next cell.

In [ ]:
# Install / upgrade brapi-py — uncomment ONE of the options below:

# Option 1: install from PyPI (once the package is published)
# %pip install -q brapi-py

# Option 2: install from local source (for development / testing)
# import subprocess, sys, pathlib
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e',
#                        str(pathlib.Path('..').resolve())])

import brapi
print('brapi-py version:', brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient
from dotenv import load_dotenv
import os

# Loads credentials from ../.env (never committed to git)
# Create a .env file in the project root with these keys:
#   BRAPI_BASE_URL=https://brapi.example.com
#   BRAPI_TOKEN_ENDPOINT=https://auth.example.com/token
#   BRAPI_CLIENT_ID=my-client
#   BRAPI_CLIENT_SECRET=my-secret
load_dotenv(dotenv_path='../.env')

client = BrapiClient(
    base_url=os.environ['BRAPI_BASE_URL'],
    token_endpoint=os.environ['BRAPI_TOKEN_ENDPOINT'],
    client_id=os.environ['BRAPI_CLIENT_ID'],
    client_secret=os.environ['BRAPI_CLIENT_SECRET'],
)
print('Client ready →', client)

## 2 — Search  (`POST /search/lists`)

Full-featured search. Handles both synchronous (HTTP 200) and asynchronous (HTTP 202 + polling) responses.

In [ ]:
# Basic search — filtered by common_crop_names()
df = (
    client.list
    .common_crop_names(['Tomatillo'])
    .search()
    .to_df()
)
print(f'{len(df)} records returned')
df.head()

In [ ]:
# Immutable builder — fork the same base query safely
base = (
    client.list
    .common_crop_names(['Tomatillo'])
)

q1_df = base.by_program_dbid(['8f5de35b']).search().to_df()
q2_df = base.by_program_name(['Better Breeding Program']).search().to_df()

print('q1:', len(q1_df), '  q2:', len(q2_df))

In [ ]:
# Bulk filter — apply multiple criteria in one call
df = (
    client.list
    .filter(
        common_crop_names=['Tomatillo'],
        program_dbids=['8f5de35b'],
        program_names=['Better Breeding Program'],  # add more filters above
    )
    .search()
    .to_df()
)
df.head(10)

## 3 — List  (`GET /lists`)

Simpler GET endpoint — same filter state, mapped to query-string params.
Useful when the server does not support the search endpoint, or for quick lookups.

In [ ]:
# GET /lists — list records
df = (
    client.list
    .common_crop_names(['Tomatillo'])
    .list()
    .to_df()
)
print(f'{len(df)} records')
df.head()

## 4 — Single-record CRUD

Get, create, update, and delete individual records.

In [ ]:
# GET /lists/ {id}  — fetch one record by primary key
LIST_DBID = 'example-001'   # ← change to a real ID

record = client.list.get_by_id(LIST_DBID)
print(record)

In [ ]:
from brapi.entities.generated_list import List

# POST /lists — create a new record
new_list = List(
    listDbId='',  # assigned by server
    listName='Example List',
)
created = client.list.create(new_list)
print('Created listDbId:', created.listDbId)

In [ ]:
# PUT /lists/ {id} — update the record created above
# (run the create cell first to populate 'created')
created.dateCreated = 'updated value'
updated = client.list.update(created.listDbId, created)
print('Updated:', updated.listDbId)

## 5 — Pipe transforms

`.pipe()` applies a pure function lazily — only one HTTP call is made regardless of how many transform stages are chained.

In [ ]:
def filter_non_null_ids(items):
    """Keep only records with a non-empty primary key."""
    return [r for r in items if r.listDbId]

def add_label(items):
    """Attach a short display label to each record."""
    for r in items:
        r.label = f'List {r.listDbId}'
    return items

df = (
    client.list
    .common_crop_names(['Tomatillo'])
    .search()
    .pipe(filter_non_null_ids)
    .pipe(add_label)
    .to_df()
)
df.head(10)

## 6 — DataFrame exploration

Useful pandas operations once you have a DataFrame.

In [ ]:
# Reload a fresh DataFrame
df = (
    client.list
    .common_crop_names(['Tomatillo'])
    .search()
    .to_df()
)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.head()

In [ ]:
# Value distribution for 'dateCreated'
if 'dateCreated' in df.columns:
    df['dateCreated'].value_counts().head(10)
else:
    print('Column not present in this result set')

In [ ]:
import json, pandas as pd

# Explode the 'data' JSON column into separate rows
col = 'data'
if col in df.columns:
    id_col = 'listDbId'
    exploded = (
        df.filter(items=[id_col, col])
        .dropna(subset=[col])
        .assign(**{col: lambda d: d[col].apply(json.loads)})
        .explode(col)
    )
    exploded.head(10)
else:
    print(f'Column {col!r} not present in this result set')

## 7 — Error handling

In [ ]:
import httpx

# 404 — record not found
try:
    client.list.get_by_id('does-not-exist-xyz')
except httpx.HTTPStatusError as e:
    print(f'HTTP {e.response.status_code}: {e.request.url}')

# Async search poll timeout
try:
    client.list.search(max_poll_attempts=1).to_list()
except TimeoutError as e:
    print('TimeoutError:', e)

# ValueError from create_many with only one item
try:
    client.list.create_many([])
except (ValueError, AttributeError) as e:
    print('Error:', e)

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print('Connection closed')